# Hyperparameter Tuning
## Telco Customer Churn Prediction

Default hyperparameters are rarely optimal. Here we use RandomizedSearchCV to find better
settings for the top two models from notebook 03.

We also do threshold tuning — moving the decision boundary below 0.5 to catch more churners.
This is a simple but effective technique that a lot of people skip.

Sections:
1. Load data and baseline models
2. Define search spaces
3. Run RandomizedSearchCV
4. Tuning results — before vs after
5. Threshold tuning
6. Cross-validation stability check
7. Save the best model

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate, learning_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

from src.preprocessing import load_data, clean_data, split_data
from src.features import engineer_features, create_preprocessor

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

## 1. Load Data and Baseline Results

In [ ]:
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = engineer_features(clean_data(load_data(RAW_PATH)))
X_train, X_test, y_train, y_test = split_data(df)

preprocessor = joblib.load("../models/preprocessor.joblib")
X_train_proc  = preprocessor.transform(X_train)
X_test_proc   = preprocessor.transform(X_test)

print(f"Train: {X_train_proc.shape}  |  churn rate: {y_train.mean():.1%}")
print(f"Test:  {X_test_proc.shape}   |  churn rate: {y_test.mean():.1%}")

In [ ]:
# load baseline models from notebook 03
baseline_xgb = joblib.load("../models/baseline_models/xgboost.joblib")
baseline_rf  = joblib.load("../models/baseline_models/random_forest.joblib")

from sklearn.metrics import f1_score, roc_auc_score

def quick_eval(model, X_test, y_test, name="model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"{name}")
    print(f"  F1: {f1_score(y_test, y_pred):.4f}  |  ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
    return f1_score(y_test, y_pred), roc_auc_score(y_test, y_prob)

print("Baseline performance (default params):")
base_xgb_f1, base_xgb_auc = quick_eval(baseline_xgb, X_test_proc, y_test, "XGBoost (baseline)")
base_rf_f1,  base_rf_auc  = quick_eval(baseline_rf,  X_test_proc, y_test, "Random Forest (baseline)")

## 2. Define Search Spaces

RandomizedSearchCV randomly samples from these distributions rather than exhaustively trying every
combination (that would be GridSearchCV and would take forever).

We try 50 random combinations with 5-fold cross-validation each, scoring on F1.
That's 250 model fits per model — enough to find something meaningfully better than default.

In [ ]:
# XGBoost search space
param_dist_xgb = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [3, 5, 7, 10],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2],
    "subsample":         [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree":  [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_alpha":         [0, 0.01, 0.1, 1.0],
    "reg_lambda":        [0, 0.01, 0.1, 1.0],
    "min_child_weight":  [1, 3, 5, 7],
    "gamma":             [0, 0.1, 0.2, 0.3],
}

# Random Forest search space
param_dist_rf = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2", None],
    "class_weight":      ["balanced", "balanced_subsample"],
}

print("XGBoost search space:")
for k, v in param_dist_xgb.items():
    print(f"  {k}: {v}")

print("\nRandom Forest search space:")
for k, v in param_dist_rf.items():
    print(f"  {k}: {v}")

## 3. Run RandomizedSearchCV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# tune XGBoost
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    verbosity=0,
    random_state=42,
)

print("Tuning XGBoost (this takes a few minutes)...")
t0 = time.time()

search_xgb = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=50,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    random_state=42,
)
search_xgb.fit(X_train_proc, y_train)

print(f"\nDone in {time.time() - t0:.0f}s")
print(f"Best F1 (CV): {search_xgb.best_score_:.4f}")
print(f"Best params:")
for k, v in search_xgb.best_params_.items():
    print(f"  {k}: {v}")

In [ ]:
# tune Random Forest
rf_base = RandomForestClassifier(n_jobs=-1, random_state=42)

print("Tuning Random Forest...")
t0 = time.time()

search_rf = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist_rf,
    n_iter=50,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    random_state=42,
)
search_rf.fit(X_train_proc, y_train)

print(f"\nDone in {time.time() - t0:.0f}s")
print(f"Best F1 (CV): {search_rf.best_score_:.4f}")
print(f"Best params:")
for k, v in search_rf.best_params_.items():
    print(f"  {k}: {v}")

## 4. Tuning Results — Before vs After

In [ ]:
tuned_xgb = search_xgb.best_estimator_
tuned_rf  = search_rf.best_estimator_

print("Tuned performance:")
tuned_xgb_f1, tuned_xgb_auc = quick_eval(tuned_xgb, X_test_proc, y_test, "XGBoost (tuned)")
tuned_rf_f1,  tuned_rf_auc  = quick_eval(tuned_rf,  X_test_proc, y_test, "Random Forest (tuned)")

In [ ]:
# before vs after bar chart
models    = ["XGBoost", "Random Forest"]
f1_before = [base_xgb_f1, base_rf_f1]
f1_after  = [tuned_xgb_f1, tuned_rf_f1]

x = np.arange(len(models))
w = 0.3
colors = sns.color_palette("Set2", 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 comparison
axes[0].bar(x - w/2, f1_before, width=w, label="Before tuning", color=colors[0], edgecolor="white")
axes[0].bar(x + w/2, f1_after,  width=w, label="After tuning",  color=colors[1], edgecolor="white")
for i, (before, after) in enumerate(zip(f1_before, f1_after)):
    axes[0].text(i - w/2, before + 0.002, f"{before:.3f}", ha="center", fontsize=9)
    axes[0].text(i + w/2, after  + 0.002, f"{after:.3f}",  ha="center", fontsize=9, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].set_ylabel("F1 Score")
axes[0].set_ylim(0, max(f1_after) * 1.15)
axes[0].set_title("F1 Score: Default vs Tuned", fontweight="bold")
axes[0].legend()

# improvement bars
improvements = [(a - b) / b * 100 for a, b in zip(f1_after, f1_before)]
bar_colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in improvements]
bars = axes[1].bar(models, improvements, color=bar_colors, edgecolor="white", width=0.4)
for bar, val in zip(bars, improvements):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.1,
                 f"+{val:.1f}%" if val >= 0 else f"{val:.1f}%",
                 ha="center", fontweight="bold")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("F1 Improvement (%)")
axes[1].set_title("Improvement from Tuning", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/fig_12_tuning_improvement.png", bbox_inches="tight")
plt.show()

for name, before, after in zip(models, f1_before, f1_after):
    print(f"{name}: F1 {before:.4f} -> {after:.4f}  ({(after-before)/before*100:+.1f}%)")

## 5. Threshold Tuning

By default sklearn classifies as Churn if predicted probability > 0.5.
But we might prefer to lower this threshold to catch more churners, at the cost of more false alarms.

This is a business decision: how much retention spend are we willing to waste on customers
who weren't actually going to leave, in exchange for catching more real churners?

The plot below shows how precision, recall, and F1 change as we move the threshold.

In [ ]:
# use the better of the two tuned models
best_tuned = tuned_xgb if tuned_xgb_f1 >= tuned_rf_f1 else tuned_rf
best_name  = "XGBoost" if tuned_xgb_f1 >= tuned_rf_f1 else "Random Forest"
print(f"Using {best_name} for threshold analysis")

from sklearn.metrics import precision_score, recall_score

y_prob = best_tuned.predict_proba(X_test_proc)[:, 1]
thresholds = np.arange(0.1, 0.91, 0.02)

precisions, recalls, f1_scores = [], [], []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))

optimal_threshold = thresholds[np.argmax(f1_scores)]
optimal_f1        = max(f1_scores)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, precisions, label="Precision", linewidth=2, color=sns.color_palette("Set2")[0])
ax.plot(thresholds, recalls,    label="Recall",    linewidth=2, color=sns.color_palette("Set2")[1])
ax.plot(thresholds, f1_scores,  label="F1 Score",  linewidth=2, color=sns.color_palette("Set2")[2])
ax.axvline(0.5, color="gray",  linestyle="--", linewidth=1.2, label="Default (0.5)")
ax.axvline(optimal_threshold, color="red", linestyle="--", linewidth=1.5,
           label=f"Optimal ({optimal_threshold:.2f})")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Precision / Recall / F1 vs Threshold  ({best_name})", fontweight="bold")
ax.legend()
ax.set_xlim(0.1, 0.9)
plt.tight_layout()
plt.savefig("../reports/fig_13_threshold_analysis.png", bbox_inches="tight")
plt.show()

print(f"Default threshold (0.50): F1 = {f1_scores[list(thresholds).index(min(thresholds, key=lambda x: abs(x-0.5)))]:.4f}")
print(f"Optimal threshold ({optimal_threshold:.2f}): F1 = {optimal_f1:.4f}")

In [ ]:
# compare a few specific thresholds
print(f"{'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
print("-" * 44)
for t in [0.30, 0.40, 0.50, optimal_threshold, 0.60]:
    y_pred_t = (y_prob >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    marker = " <-- optimal" if abs(t - optimal_threshold) < 0.01 else ""
    marker = " <-- default" if abs(t - 0.5) < 0.01 else marker
    print(f"{t:>10.2f}  {p:>10.4f}  {r:>8.4f}  {f:>8.4f}{marker}")

## 6. Cross-Validation Stability Check

A good model should perform consistently across different data splits, not just on one particular test set.
High standard deviation across folds means the model is sensitive to which data it sees — not great.


In [ ]:
print(f"Running 5-fold cross-validation on {best_name}...")

cv_results = cross_validate(
    best_tuned,
    X_train_proc,
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    n_jobs=-1,
)

print(f"\n5-Fold CV Results ({best_name} — tuned):")
print(f"{'Metric':>12}  {'Mean':>8}  {'Std':>8}  {'Min':>8}  {'Max':>8}")
print("-" * 52)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    scores = cv_results[f"test_{metric}"]
    print(f"{metric:>12}  {scores.mean():.4f}  {scores.std():.4f}  {scores.min():.4f}  {scores.max():.4f}")

In [ ]:
# box plot of CV scores across folds
fig, ax = plt.subplots(figsize=(10, 5))

metrics_to_plot = ["accuracy", "precision", "recall", "f1", "roc_auc"]
data_to_plot    = [cv_results[f"test_{m}"] for m in metrics_to_plot]

bp = ax.boxplot(data_to_plot, patch_artist=True, notch=False,
                medianprops={"color": "black", "linewidth": 2})
colors = sns.color_palette("Set2", len(metrics_to_plot))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)

ax.set_xticklabels([m.title() for m in metrics_to_plot])
ax.set_ylabel("Score")
ax.set_title(f"CV Score Distribution Across 5 Folds ({best_name})", fontweight="bold")
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("../reports/fig_14_cv_stability.png", bbox_inches="tight")
plt.show()

f1_std = cv_results["test_f1"].std()
if f1_std < 0.02:
    print(f"F1 std = {f1_std:.4f} — model is stable across folds.")
else:
    print(f"F1 std = {f1_std:.4f} — some variance across folds, worth investigating.")

## 7. Save the Best Model

In [ ]:
os.makedirs("../models/tuned_models", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# save both tuned models
joblib.dump(tuned_xgb, "../models/tuned_models/xgboost_tuned.joblib")
joblib.dump(tuned_rf,  "../models/tuned_models/random_forest_tuned.joblib")

# save the best one as the production model
joblib.dump(best_tuned, "../models/best_model.joblib")

# save the optimal threshold alongside it
import json
model_meta = {
    "model_name":         best_name,
    "optimal_threshold":  round(float(optimal_threshold), 2),
    "tuned_f1_test":      round(float(optimal_f1), 4),
    "tuned_roc_auc_test": round(float(tuned_xgb_auc if best_name == "XGBoost" else tuned_rf_auc), 4),
    "cv_f1_mean":         round(float(cv_results["test_f1"].mean()), 4),
    "cv_f1_std":          round(float(cv_results["test_f1"].std()), 4),
}
with open("../models/model_meta.json", "w") as f:
    json.dump(model_meta, f, indent=2)

print("Saved:")
print(f"  models/tuned_models/xgboost_tuned.joblib")
print(f"  models/tuned_models/random_forest_tuned.joblib")
print(f"  models/best_model.joblib  ({best_name})")
print(f"  models/model_meta.json")
print(f"\nBest model summary:")
for k, v in model_meta.items():
    print(f"  {k}: {v}")